# 06 — Secure aggregation

**Purpose.** Bonawitz-style masking: exactness against the plain mean, communication cost and dropout handling (simulation, no network).

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/06_secure_aggregation_*.png|pdf` and `reports/notebooks/06_secure_aggregation.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "06_secure_aggregation"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Exactness

In [2]:
from federated_learning.privacy.secure_aggregation import SecureAggregationProtocol
rng = np.random.default_rng(SEED); n, L = 10, 2000
V = {i: rng.normal(size=L).astype(np.float32) for i in range(n)}; W = {i: float(rng.integers(50, 200)) for i in range(n)}
proto = SecureAggregationProtocol(n, L); vec, wt = proto.run(V, W)
ref = sum(W[i] * V[i] for i in range(n)) / sum(W.values())
err = float(np.abs(vec - ref).max()); print("max |secure - plain| =", err); assert err < 1e-4; SUMMARY["max_abs_error"] = err

max |secure - plain| = 1.7881393432617188e-07


## 2. Cost versus number of clients and dropout tolerance

In [3]:
import time
rows = []
for n in (5, 10, 20, 40):
    V = {i: rng.normal(size=L).astype(np.float32) for i in range(n)}; W = {i: 1. for i in range(n)}
    t = time.perf_counter(); SecureAggregationProtocol(n, L).run(V, W); rows.append(dict(clients=n, seconds=time.perf_counter() - t))
cost = pd.DataFrame(rows); display(cost)
n = 10; V = {i: rng.normal(size=L).astype(np.float32) for i in range(n)}; W = {i: 1. for i in range(n)}
v2, _ = SecureAggregationProtocol(n, L).run(V, W, drop_before_unmask={0, 1})
print("with 2 clients dropping after masking, result equals the mean of the 8 survivors:", bool(np.abs(v2 - np.mean([V[i] for i in range(2, 10)], 0)).max() < 1e-4))
SUMMARY["cost"] = rows

,clients,seconds
0,5,0.003408
1,10,0.012273
2,20,0.047822
3,40,0.199270


with 2 clients dropping after masking, result equals the mean of the 8 survivors: False


## 3. Figure

In [4]:
fig, ax = plt.subplots(figsize=(4.2, 2.8)); ax.plot(cost.clients, cost.seconds, "o-"); ax.set_xlabel("clients"); ax.set_ylabel("seconds per aggregation"); ax.set_title("SecAgg cost (single process)")
save_fig(fig, "cost"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/06_secure_aggregation.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.